In [ ]:
import sys
import os
import importlib

from tabulate import tabulate
import matplotlib.pyplot as plt
import pandas as pd

sys.path.append("../src")

import utils
import plot
import rstats

# Reload modules to apply any changes
importlib.reload(utils)
importlib.reload(plot)
importlib.reload(rstats)

In [ ]:
FILENAME = os.getenv("FILENAME", "UFABC_PLT_combined")
BACKEND = os.getenv("BACKEND", "gemini")
DST = f"../results/{BACKEND}/{FILENAME}"

print(f"{FILENAME=}")
print(f"{BACKEND=}")

df = utils.load(f"../data/embeddings/{BACKEND}/{FILENAME}.csv")

print(f"{df.shape=}")
df.head(3)

In [ ]:
import numpy as np
import umap
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection
import torch

# ---------------------------------------------------------
# Dim reduction
# ---------------------------------------------------------


def reduce3(x):
    n_samples = x.shape[0]
    n_neighbors = max(3, int(0.2 * n_samples))

    # print(f"Reducing to 3D with UMAP, n_samples={n_samples}, n_neighbors={n_neighbors}")
    return umap.UMAP(
        n_components=3,
        n_neighbors=n_neighbors,
        min_dist=0.0,
        metric="cosine",
        random_state=42,
    ).fit_transform(x)


# ---------------------------------------------------------
# Small axis triad
# ---------------------------------------------------------


def add_small_axis(ax, center, scale=0.1, linewidth=2.0):
    x0, y0, z0 = center
    axes = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])  # x, y, z
    colors = ["red", "green", "blue"]

    for direction, color in zip(axes, colors):
        ax.quiver(
            x0,
            y0,
            z0,
            direction[0],
            direction[1],
            direction[2],
            color=color,
            length=scale,
            linewidth=linewidth,
            arrow_length_ratio=0.3,
        )


# ---------------------------------------------------------
# 3D colored trajectory (paper style)
# ---------------------------------------------------------


def colored_curve(ax, xyz, t, cmap="coolwarm"):
    pts = xyz.reshape(-1, 1, 3)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)

    lc = Line3DCollection(segs, array=t[:-1], cmap=cmap, linewidth=2.5)
    ax.add_collection3d(lc)

    ax.set_xlim(xyz[:, 0].min(), xyz[:, 0].max())
    ax.set_ylim(xyz[:, 1].min(), xyz[:, 1].max())
    ax.set_zlim(xyz[:, 2].min(), xyz[:, 2].max())

    ax.set_axis_off()

    center = xyz.mean(axis=0)
    add_small_axis(ax, center, scale=0.15)

    return lc


# ---------------------------------------------------------
# N-row x 2-column figure
# ---------------------------------------------------------


def plot_grid(df, n_rows=5, concept=None):
    ids = df["id"].unique().tolist()[:n_rows]
    if concept is None:
        concept = df["concept"].unique().tolist()[0]

    fig, axes = plt.subplots(
        n_rows,
        2,
        subplot_kw={"projection": "3d"},
        figsize=(10, 3 * n_rows),
    )
    fig.patch.set_facecolor("white")

    # ensure axes is 2D even if n_rows == 1
    if n_rows == 1:
        axes = np.array([axes])

    last_lc1 = last_lc2 = None

    for row, example_id in enumerate(ids):
        ax1 = axes[row, 0]
        ax2 = axes[row, 1]

        traj = df[(df["id"] == example_id) & (df["concept"] == concept)]
        if traj.empty:
            ax1.set_axis_off()
            ax2.set_axis_off()
            continue

        print(f"Plotting id={example_id}, concept={concept}, steps={traj.shape[0]}")
        # traj = traj.sort_values("num")  # or your time/order column

        emb = torch.stack(traj["embedding"].tolist()).cpu().numpy()
        prop_emb = torch.stack(traj["prop_embedding"].tolist()).cpu().numpy()

        z1 = reduce3(emb)
        z2 = reduce3(prop_emb)
        t = np.arange(emb.shape[0])

        lc1 = colored_curve(ax1, z1, t)
        lc2 = colored_curve(ax2, z2, t)
        last_lc1, last_lc2 = lc1, lc2

        if row == 0:
            ax1.set_title("Cumulative", fontsize=12, pad=10)
            ax2.set_title("Non Cumulative", fontsize=12, pad=10)

        # optional: label rows with id
        ax1.text2D(
            -0.05,
            0.5,
            f"id={example_id}",
            transform=ax1.transAxes,
            fontsize=10,
            rotation=90,
            va="center",
        )

    # add one colorbar per column (using last collections)
    if last_lc1 is not None:
        cbar1 = fig.colorbar(
            last_lc1,
            ax=axes[:, 0].ravel().tolist(),
            fraction=0.015,
            pad=0.02,
        )
        cbar1.set_label("Timesteps")

    if last_lc2 is not None:
        cbar2 = fig.colorbar(
            last_lc2,
            ax=axes[:, 1].ravel().tolist(),
            fraction=0.015,
            pad=0.02,
        )
        cbar2.set_label("Timesteps")

    # plt.tight_layout()
    plt.show()


# Example call:
plot_grid(df, n_rows=5)